# Inspect IndicCorp V2 parquet

Inspect the parquet shards produced by `dataset.indiccorp_dataset` (IndicCorp V2 Hindi `.txt` → parquet).

Open with kernel cwd = **repo root** or **`notebook/`** — repo is auto-detected. Run cells top to bottom.

Checks performed:
- shard listing + sizes
- schema parity against Sangraha `verified/hin` eval parquet (`doc_id`, `text`, `type`, all `large_string`)
- row counts, sample documents, text length stats, `doc_id` uniqueness
- `ParquetMLMDataset` round-trip (the exact reader used in pretraining)

In [6]:
from pathlib import Path
import sys

from pipeline_steps import ensure_src_on_path

REPO = ensure_src_on_path()

# Where the conversion script writes shards by default; override if needed.
PARQUET_DIR = REPO / "data" / "indiccorp_v2" / "parquet" / "hi"
REF_PARQUET = REPO / "data" / "eval" / "hi" / "verified" / "hin" / "data-15.parquet"

print("repo       :", REPO)
print("parquet dir:", PARQUET_DIR, "(exists)" if PARQUET_DIR.exists() else "(MISSING)")

repo       : /home/krrish/Desktop/Programming/indic-modernBERT
parquet dir: /home/krrish/Desktop/Programming/indic-modernBERT/data/indiccorp_v2/parquet/hi (exists)


## 1 — Shards on disk

In [7]:
shards = sorted(PARQUET_DIR.glob("*.parquet"))
assert shards, f"No parquet shards under {PARQUET_DIR}. Run: uv run python -m dataset.indiccorp_dataset"

total_bytes = sum(p.stat().st_size for p in shards)
print(f"{len(shards)} shard(s), {total_bytes / 1e6:.1f} MB total\n")
for p in shards[:10]:
    print(f"  {p.name:24s} {p.stat().st_size / 1e6:8.2f} MB")
if len(shards) > 10:
    print(f"  ... (+{len(shards) - 10} more)")

356 shard(s), 15815.5 MB total

  hi-1-00000.parquet          44.72 MB
  hi-1-00001.parquet          44.69 MB
  hi-1-00002.parquet          44.75 MB
  hi-1-00003.parquet          44.66 MB
  hi-1-00004.parquet          44.58 MB
  hi-1-00005.parquet          44.73 MB
  hi-1-00006.parquet          44.48 MB
  hi-1-00007.parquet          44.37 MB
  hi-1-00008.parquet          44.67 MB
  hi-1-00009.parquet          44.52 MB
  ... (+346 more)


## 2 — Schema parity with Sangraha eval parquet

In [8]:
import pyarrow.parquet as pq

got = pq.ParquetFile(shards[0]).schema_arrow
print("IndicCorp schema:")
for f in got:
    print(f"  {f.name:8s} {f.type}")

if REF_PARQUET.exists():
    ref = pq.ParquetFile(REF_PARQUET).schema_arrow
    print("\nSangraha reference schema:")
    for f in ref:
        print(f"  {f.name:8s} {f.type}")
    assert ref.equals(got), "Schema mismatch vs Sangraha verified/hin!"
    print("\n✓ schema matches Sangraha verified/hin exactly")
else:
    print("\n⊘ reference parquet not found; skipping parity check")

IndicCorp schema:
  doc_id   large_string
  text     large_string
  type     large_string

Sangraha reference schema:
  doc_id   large_string
  text     large_string
  type     large_string

✓ schema matches Sangraha verified/hin exactly


## 3 — Row counts + sample documents

In [9]:
total_rows = sum(pq.read_metadata(p).num_rows for p in shards)
print(f"total documents: {total_rows:,}\n")

sample = pq.read_table(shards[0]).slice(0, 5).to_pylist()
for i, row in enumerate(sample):
    print(f"[{i}] type={row['type']!r} doc_id={row['doc_id']}")
    print(f"    {row['text'][:200]}")
    print()

total documents: 70,923,978

[0] type='indiccorp_v2' doc_id=e516e19325a33a43644669c5272fd90049fe3a77
    लोगों को बिलों संबंधी सुविधा देना ही उनका काम

[1] type='indiccorp_v2' doc_id=65346a77f865b6e1f49bcdec1c5d1a37abe7ccdd
    इनेलो 1987 में उस वक्त ऐसे ही दोराहे पर खड़ी थी, जब पूर्व उपप्रधानमंत्री देवीलाल ने अपने पुत्र ओमप्रकाश चौटाला को अपना राजनीतिक उत्तराधिकारी घोषित किया था। हालांकि तब पार्टी पर देवीलाल की मजबूत पकड़ क

[2] type='indiccorp_v2' doc_id=94ad5e2834e5dc1c03048e695ef77cf3f87b525c
    जहां आई थी तबाही उस घाटी क्षेत्र में खतरा ज्यादा

[3] type='indiccorp_v2' doc_id=401b02be653f9bfbc7825be2362370a202dc0efe
    इसके बाद केंद्र की ओर से प्रदेश सरकार को पीएमजीएसवाई में 200 करोड़ रुपये की राशि उपलब्ध करा दी गई। भाजपा के मीडिया प्रभारी दिवाकर सिंह ने शनिवार को बताया कि केंद्र ने प्रदेश सरकार को 200 करोड़ रुपये भ

[4] type='indiccorp_v2' doc_id=eb0db5e48a8fffc73d8e1e152fa31115a190a5bd
    यह पूछने पर कि इस बड़े मैच से पहले उनकी नींद गायब हुई तो बाबर ने कहा, "हम काफी टूर्नामेंट 

## 4 — Text length stats + doc_id uniqueness

In [10]:
import statistics

lengths: list[int] = []
doc_ids: list[str] = []
empties = 0

number_shard = 10
current_shard = 0

for p in shards:
    table = pq.read_table(p, columns=["doc_id", "text"])
    for text in table.column("text").to_pylist():
        text = text or ""
        lengths.append(len(text))
        if not text.strip():
            empties += 1
    doc_ids.extend(table.column("doc_id").to_pylist())

    current_shard +=1 

    if current_shard > number_shard : 
        break 


print(f"documents      : {len(lengths):,}")
print(f"empty text     : {empties}")
print(f"char length    : min={min(lengths)} max={max(lengths)} "
      f"mean={statistics.mean(lengths):.0f} median={statistics.median(lengths):.0f}")
print(f"doc_id total   : {len(doc_ids):,}")
print(f"doc_id unique  : {len(set(doc_ids)):,}")
dup = len(doc_ids) - len(set(doc_ids))
print(f"doc_id dupes   : {dup}  (duplicates == identical repeated text, expected)")

documents      : 2,200,000
empty text     : 0
char length    : min=4 max=352657 mean=293 median=214
doc_id total   : 2,200,000
doc_id unique  : 2,198,227
doc_id dupes   : 1773  (duplicates == identical repeated text, expected)


## 5 — ParquetMLMDataset round-trip

Read the shards through the exact dataset class used by pretraining (mmap row-group reads, `text` column).

In [11]:
from pretrain.parquet_mlm import ParquetMLMDataset

ds = ParquetMLMDataset(PARQUET_DIR, text_column="text", use_script_norm=False)
print("ParquetMLMDataset len:", len(ds))
print("first:", repr(ds[0][:120]))
print("last :", repr(ds[len(ds) - 1][:120]))
assert len(ds) == total_rows, "Dataset length disagrees with parquet metadata"
print("\n✓ ParquetMLMDataset reads all", len(ds), "documents")

ParquetMLMDataset len: 70923978
first: 'लोगों को बिलों संबंधी सुविधा देना ही उनका काम'
last : 'वह तेजी से बढ़ा भी'

✓ ParquetMLMDataset reads all 70923978 documents
